In [31]:
import torch
import torch.nn as nn
import torchbearer
from torchbearer import Trial
import math
import numpy as np
from torch.utils.data import Dataset, DataLoader

# 1. Modified LCM Model with TorchBearer compatibility
class LCMTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=3, max_num=10000):
        super().__init__()
        self.embedding = nn.Embedding(max_num, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead)
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        
        self.fc = nn.Linear(d_model, max_num)
        self.max_num = max_num
        
    def forward(self, x, state=None):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

# 2. Data Generation with Safe Pairs
class LCMDataset(Dataset):
    def __init__(self, max_num=10000, num_samples=10000, seed=42):
        self.max_num = max_num
        self.rng = np.random.RandomState(seed)
        self.data = []
        
        for _ in range(num_samples):
            lcm = self.rng.randint(1, max_num)
            divisors = [d for d in range(1, lcm+1) if lcm % d == 0]
            
            for _ in range(100):  # Try 100 combinations
                a, b = self.rng.choice(divisors, size=2)
                if math.lcm(a, b) == lcm:
                    self.data.append(([a, b], lcm))
                    break
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        inputs, target = self.data[idx]
        return (
            torch.LongTensor(inputs),  # Shape [2]
            torch.LongTensor([target]).squeeze()  # Shape [] (scalar)
        )

# 4. Positional Encoding (Required for Transformer)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


In [35]:
# 3. TorchBearer Trial Setup

# Config
max_num = 100
batch_size = 64
d_model = 128

# Data
train_set = LCMDataset(max_num=max_num, num_samples=50000)
val_set = LCMDataset(max_num=max_num, num_samples=10000)
test_set = LCMDataset(max_num=max_num, num_samples=10000)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(train_set, batch_size=batch_size)

# Model
model = LCMTransformer(max_num=max_num, d_model=d_model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# TorchBearer Trial
trial = Trial(
    model,
    optimizer,
    criterion,
    metrics=['acc', 'loss'],
    callbacks=[
        torchbearer.callbacks.MostRecent('model.pt'),
        torchbearer.callbacks.EarlyStopping(patience=3)
    ]
).to('cuda' if torch.cuda.is_available() else 'cpu')

trial.with_generators(
    train_generator=train_loader,
    val_generator=val_loader,
    test_generator=test_loader
)

# Training
trial.run(epochs=10)

# Evaluation
results = trial.evaluate(data_key=torchbearer.TEST_DATA)
print(f"Test Accuracy: {results['test_acc']:.2%}")


c:\Users\xw3g19\.conda\envs\deeplearning\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


0/10(t):   0%|          | 0/782 [00:00<?, ?it/s]

0/10(v):   0%|          | 0/157 [00:00<?, ?it/s]

1/10(t):   0%|          | 0/782 [00:00<?, ?it/s]

1/10(v):   0%|          | 0/157 [00:00<?, ?it/s]

...

8/10(v):   0%|          | 0/157 [00:00<?, ?it/s]

9/10(t):   0%|          | 0/782 [00:00<?, ?it/s]

9/10(v):   0%|          | 0/157 [00:00<?, ?it/s]

0/1(e):   0%|          | 0/782 [00:00<?, ?it/s]

Test Accuracy: 92.55%


In [ ]:
trial.predict()

0/1(p):   0%|          | 0/157 [00:00<?, ?it/s]

torch.Size([10000, 2])